# Network Security Log Analyzer

A Python tool that parses network login logs, detects suspicious activity (failed login spikes), and identifies possible brute force attacks. Generates a timestamped security report.

**Author:** Asmaa  
**Tech:** Python (no external libraries required)

## Step 1: Create Sample Log Data

This simulates a real network log file. In production, this would be an actual log file from a server, firewall, or device.

In [ ]:
log_data = """2026-06-18 08:01:12 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:15 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:18 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:20 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:22 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:25 192.168.1.1 FAILED LOGIN user=admin
2026-06-18 08:01:30 192.168.1.1 SUCCESS LOGIN user=admin
2026-06-18 08:03:10 192.168.1.5 FAILED LOGIN user=root
2026-06-18 08:03:15 192.168.1.5 FAILED LOGIN user=root
2026-06-18 09:00:00 10.0.0.2 SUCCESS LOGIN user=asmaa
2026-06-18 09:05:00 10.0.0.3 SUCCESS LOGIN user=sara"""

with open("sample_log.txt", "w") as f:
    f.write(log_data)

print("Log file created!")

## Step 2: Parse the Log and Analyze

- Reads the log line by line
- Tracks failed login attempts per IP address
- Tracks successful logins
- Flags any IP exceeding the failure threshold
- Detects brute force attacks (failed attempts followed by a success from the same IP)

In [ ]:
failed_logins = {}
successful_logins = []
threshold = 5  # alert if an IP fails this many times or more

with open("sample_log.txt", "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 6:
            continue

        date = parts[0]
        time = parts[1]
        ip = parts[2]
        status = parts[3]
        username = parts[5].split("=")[1]

        if status == "FAILED":
            if ip not in failed_logins:
                failed_logins[ip] = {"count": 0, "usernames": set(), "times": []}
            failed_logins[ip]["count"] += 1
            failed_logins[ip]["usernames"].add(username)
            failed_logins[ip]["times"].append(date + " " + time)

        elif status == "SUCCESS":
            successful_logins.append({"ip": ip, "user": username, "time": date + " " + time})

# Brute force detection: IP crossed threshold AND eventually succeeded
brute_force_ips = []
successful_ips = [login["ip"] for login in successful_logins]
for ip, data in failed_logins.items():
    if data["count"] >= threshold and ip in successful_ips:
        brute_force_ips.append(ip)

print("Analysis complete.")

## Step 3: Print the Security Report

In [ ]:
print("=" * 45)
print("   NETWORK SECURITY ALERT REPORT")
print("=" * 45)

print("\nFAILED LOGIN ANALYSIS:")
for ip, data in failed_logins.items():
    if data["count"] >= threshold:
        print("\n  ALERT - Suspicious IP: " + ip)
        print("  Failed attempts : " + str(data["count"]))
        print("  Targeted users  : " + ", ".join(data["usernames"]))
        print("  First attempt   : " + data["times"][0])
        print("  Last attempt    : " + data["times"][-1])
    else:
        print("\n  OK - IP: " + ip + " - " + str(data["count"]) + " failed attempts")

print("\nSUCCESSFUL LOGINS:")
for login in successful_logins:
    print("  " + login["user"] + " from " + login["ip"] + " at " + login["time"])

print("\n" + "=" * 45)
print("BRUTE FORCE DETECTION:")
if brute_force_ips:
    for ip in brute_force_ips:
        print("\n  CRITICAL - Brute force SUCCESS from: " + ip)
        print("  " + str(failed_logins[ip]["count"]) + " failed attempts then logged in successfully")
        print("  ACTION: Block this IP immediately")
else:
    print("  No brute force attacks detected")

print("\n" + "=" * 45)
print("  Total IPs monitored  : " + str(len(failed_logins)))
print("  Suspicious IPs       : " + str(sum(1 for d in failed_logins.values() if d["count"] >= threshold)))
print("  Brute force detected : " + str(len(brute_force_ips)))
print("  Successful logins    : " + str(len(successful_logins)))
print("=" * 45)

## Step 4: Export Report to a Text File

Saves a timestamped `.txt` report — useful for record-keeping, audits, or sharing with a team.

In [ ]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
report_filename = "security_report_" + timestamp + ".txt"

with open(report_filename, "w") as report:
    report.write("=" * 45 + "\n")
    report.write("   NETWORK SECURITY ALERT REPORT\n")
    report.write("   Generated: " + timestamp + "\n")
    report.write("=" * 45 + "\n")

    report.write("\nFAILED LOGIN ANALYSIS:\n")
    for ip, data in failed_logins.items():
        if data["count"] >= threshold:
            report.write("\n  ALERT - Suspicious IP: " + ip + "\n")
            report.write("  Failed attempts : " + str(data["count"]) + "\n")
            report.write("  Targeted users  : " + ", ".join(data["usernames"]) + "\n")
            report.write("  First attempt   : " + data["times"][0] + "\n")
            report.write("  Last attempt    : " + data["times"][-1] + "\n")
        else:
            report.write("\n  OK - IP: " + ip + " - " + str(data["count"]) + " failed attempts\n")

    report.write("\nSUCCESSFUL LOGINS:\n")
    for login in successful_logins:
        report.write("  " + login["user"] + " from " + login["ip"] + " at " + login["time"] + "\n")

    report.write("\n" + "=" * 45 + "\n")
    report.write("BRUTE FORCE DETECTION:\n")
    if brute_force_ips:
        for ip in brute_force_ips:
            report.write("\n  CRITICAL - Brute force SUCCESS from: " + ip + "\n")
            report.write("  " + str(failed_logins[ip]["count"]) + " failed attempts then logged in successfully\n")
            report.write("  ACTION: Block this IP immediately\n")
    else:
        report.write("  No brute force attacks detected\n")

    report.write("\n" + "=" * 45 + "\n")
    report.write("  Total IPs monitored  : " + str(len(failed_logins)) + "\n")
    report.write("  Suspicious IPs       : " + str(sum(1 for d in failed_logins.values() if d["count"] >= threshold)) + "\n")
    report.write("  Brute force detected : " + str(len(brute_force_ips)) + "\n")
    report.write("  Successful logins    : " + str(len(successful_logins)) + "\n")
    report.write("=" * 45 + "\n")

print("Report saved as: " + report_filename)